# Local validation of the Databricks export

Runs the automated checks on every series in `input_output/output_databricks/` — the raw
hourly copy pulled from Databricks via git. The export itself is never modified.

**Flag layers** run in order on `head`. Each stores the original value of what it flags in
its own column and sets `head` to NaN, so the next layer only sees what is left:

| | Check | Default |
|---|---|---|
| `v1` | Out of bounds: below the filter bottom, or above the top of the well | `eind_filter` / `bovenkant_peilbuis` |
| `v2` | Unrealistic step change between two hours | up > +0.30 m, down < −0.05 m |
| `v3` | At the filter bottom: within 0.15 m of `eind_filter` (inclusive) | −4.10 → −3.95 and −4.05 flagged, −3.92 not |
| `v4` | Statistical outlier (Tukey / IQR) | outside Q1 − 1.5·IQR .. Q3 + 1.5·IQR |

**Autoflag rules**, on the raw value (they act on the review store):

| | Rule | Default |
|---|---|---|
| `rule_a` | Drop ≥ 0.10 m in one hourly step (contiguous hours) | report-only |
| `rule_b` | Value ≤ filter bottom + 0.10 m | applied (`afgekeurd = 1` in the store) |

**Manual review** always overrides these: `checks.apply_autoflags()` never touches a row a
human marked `gecontroleerd`.

Thresholds and paths come from `config/settings.yaml`. Output (gitignored, rebuilt each run):
`output_data/validated/<filename>.csv` and `output_data/validation_summary.csv`.

In [ ]:
import sys
from pathlib import Path

# Make the repo importable wherever Jupyter was started (repo root or local_validation/).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "local_validation").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from local_validation import checks, data_io

settings = data_io.load_settings()
TH = data_io.thresholds(settings)
TH

## The export

In [ ]:
meta = data_io.load_object_data(settings)
print(f"{len(meta)} series")
print(meta["vendor"].value_counts().to_string())
if "matched" in meta:
    print()
    print(meta["matched"].value_counts().to_string())
print()
print(f"eind_filter known:  {meta['eind_filter'].notna().sum()}  (v1 lower bound, v3, rule B)")
print(f"top of well known:  {meta['bovenkant_peilbuis'].notna().sum()}  (v1 upper bound)")

## One series

Pick any `filename` from `object_data.csv`. The plot shows the raw series in grey and each
flagged hour in the colour of the layer that took it.

In [ ]:
FILENAME = meta.loc[meta["eind_filter"].notna(), "filename"].iloc[0]   # or e.g. "HB15-PB1__b4f0f98e8948"

row = next(meta[meta["filename"] == FILENAME].itertuples(index=False))
result = data_io.validate_row(settings, row, TH)
pd.Series(data_io.summarize(row, result))

In [ ]:
def plot_result(result, row, ax=None):
    ax = ax or plt.subplots(figsize=(14, 4.5))[1]
    ax.plot(result.index, result["raw_head"], color="0.75", lw=0.8, label="raw")
    colours = {"v1": "tab:red", "v2": "tab:orange", "v3": "tab:purple", "v4": "tab:blue"}
    for col, colour in colours.items():
        hit = result[col].dropna()
        if len(hit):
            ax.scatter(hit.index, hit, s=6, color=colour, label=f"{col} ({len(hit)})", zorder=3)
    rb = result.loc[result["rule_b"], "raw_head"]
    if len(rb):
        ax.scatter(rb.index, rb, s=18, facecolors="none", edgecolors="black",
                   label=f"rule B ({len(rb)})", zorder=4)
    eind = checks.as_float(row.eind_filter)
    if eind is not None:
        ax.axhline(eind, color="tab:purple", ls="--", lw=1, label="eind_filter")
        ax.axhspan(eind - TH.v3_band, eind + TH.v3_band, color="tab:purple", alpha=0.08)
    top = checks.as_float(row.bovenkant_peilbuis)
    if top is not None:
        ax.axhline(top, color="tab:red", ls=":", lw=1, label="top of well")
    ax.set_title(f"{row.filename}  ({row.vendor})")
    ax.set_ylabel("head [m NAP]")
    ax.legend(loc="upper left", fontsize=8, ncol=4)
    return ax


plot_result(result, row);

## All series

Validates everything and writes `output_data/validated/` + `output_data/validation_summary.csv`.
About 30 s for ~700 series.

In [ ]:
summary = data_io.validate_all(settings, TH)
totals = summary[[c for c in summary if c.startswith("n_")]].sum()
pd.DataFrame({"hours": totals, "pct": (100 * totals / totals["n_hours"]).round(2)})

In [ ]:
by_vendor = summary.groupby("vendor")[[c for c in summary if c.startswith("n_")]].sum()
by_vendor["pct_kept"] = (100 * by_vendor["n_valid_after_flags"] / by_vendor["n_hours"]).round(1)
by_vendor

Series that lose the largest share of their hours — the first ones to look at in review.

In [ ]:
flag_cols = [f"n_{c}" for c in checks.FLAG_COLUMNS]
worst = summary.assign(pct_flagged=(100 * (1 - summary["n_valid_after_flags"] / summary["n_hours"])).round(1))
worst.sort_values("pct_flagged", ascending=False)[["filename", "vendor", "n_hours", *flag_cols,
                                                    "n_rule_b", "pct_flagged"]].head(15)

## Next: manual review

The Streamlit review (still to be rebuilt from `docs/archive/`) reads
`output_data/validated/` as hints, seeds each review store with rule B through
`checks.apply_autoflags()`, and writes the verdicts to `input_output/output_repository/`.
Push those, pull in Databricks, and run `03_validated_to_delta`.